Cell 1: Setup & Configuration

In [0]:
# MAGIC %md
# MAGIC # ⚡ EV Site Selection — NB01: Data Ingestion
# MAGIC **Layer:** Bronze | **Objective:** ดึงข้อมูล POI จาก OSM API และสกัดค่าจาก GeoTIFF เข้า Delta Lake

%pip install rasterio

import os
import time
import requests
import pandas as pd
import numpy as np
import rasterio
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

DATABASE_NAME = "ev_site_selection"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")

# Bounding Box: กรุงเทพฯ + ปริมณฑล
BBOX = {"south": 13.45, "west": 100.25, "north": 14.05, "east": 100.95}

# Grid ขนาด ~500x500 เมตร
GRID_LAT_SIZE = 0.0045
GRID_LON_SIZE = 0.0051

print(f"✅ Setup Complete | Database: {DATABASE_NAME}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Setup Complete | Database: ev_site_selection


Cell 2: Helper Functions

In [0]:
# MAGIC %md
# MAGIC ## Cell 2: Helper Functions

def query_overpass_single(bbox, tag_key, tag_values):
    """Query OSM Overpass API ทีละ tag พร้อม retry สูงสุด 3 รอบ"""
    bbox_str = f"{bbox['south']},{bbox['west']},{bbox['north']},{bbox['east']}"

    query_parts = []
    if isinstance(tag_values, list) and len(tag_values) > 0:
        for item in tag_values:
            query_parts.append(f'node["{tag_key}"="{item}"]({bbox_str});')
    else:
        query_parts.append(f'node["{tag_key}"]({bbox_str});')

    overpass_query = f"[out:json][timeout:180];({''.join(query_parts)});out body;"
    url     = "http://overpass-api.de/api/interpreter"
    headers = {"User-Agent": "EVSiteOptimizer/1.0"}

    for attempt in range(3):
        response = requests.post(url, data={"data": overpass_query}, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code in [504, 429]:
            print(f"   ⚠️ Server busy (attempt {attempt+1}/3) — retrying in 10s...")
            time.sleep(10)
        else:
            raise Exception(f"OSM API Error {response.status_code}: {response.text}")
    return None


def process_to_list(osm_json):
    """แปลง OSM JSON response เป็น list of POI records"""
    if not osm_json or "elements" not in osm_json:
        return []

    poi_list = []
    for element in osm_json.get("elements", []):
        tags     = element.get("tags", {})
        poi_type = "other"
        if   tags.get("shop")    == "mall":               poi_type = "shopping_mall"
        elif tags.get("amenity") == "charging_station":   poi_type = "charging_station"
        elif tags.get("amenity") == "fuel":               poi_type = "fuel_station"
        elif tags.get("tourism") == "hotel":              poi_type = "hotel"
        elif "office" in tags:                            poi_type = "office_building"

        poi_list.append({
            "osm_id":   str(element["id"]),
            "lat":      float(element["lat"]),
            "lon":      float(element["lon"]),
            "poi_type": poi_type,
            "name":     tags.get("name", "Unknown"),
        })
    return poi_list

print("✅ Helper Functions Loaded")

✅ Helper Functions Loaded


Cell 3: Ingest OSM POI + EV Station

In [0]:
# MAGIC %md
# MAGIC ## Cell 3: Ingest OSM POI + EV Stations

poi_mapping = {
    "shop":    ["mall"],
    "amenity": ["charging_station", "fuel", "parking"],
    "tourism": ["hotel"],
    "office":  [],
}

print("⏳ Fetching POI data from OSM...")
all_poi_records = []

for k, v in poi_mapping.items():
    print(f"   → Fetching tag: '{k}'")
    try:
        osm_data = query_overpass_single(BBOX, k, v)
        records  = process_to_list(osm_data)
        all_poi_records.extend(records)
        print(f"     ✅ {len(records):,} nodes retrieved")
        time.sleep(5)
    except Exception as e:
        print(f"     ❌ Failed for '{k}': {e}")

if all_poi_records:
    df_osm = spark.createDataFrame(pd.DataFrame(all_poi_records)) \
                  .withColumn("ingested_at", current_timestamp())
    df_osm.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{DATABASE_NAME}.bronze_osm_poi")
    print(f"\n📦 Saved {df_osm.count():,} POI records → bronze_osm_poi")
    display(df_osm.groupBy("poi_type").count())
else:
    print("❌ No data retrieved — check network or OSM server status")

⏳ Fetching POI data from OSM...
   → Fetching tag: 'shop'
     ✅ 27 nodes retrieved
   → Fetching tag: 'amenity'
     ✅ 841 nodes retrieved
   → Fetching tag: 'tourism'
     ✅ 692 nodes retrieved
   → Fetching tag: 'office'
     ✅ 893 nodes retrieved

📦 Saved 2,453 POI records → bronze_osm_poi


poi_type,count
other,310
fuel_station,472
shopping_mall,27
charging_station,59
hotel,692
office_building,893


Cell  4: Ingest WorldPop Population

In [0]:
# MAGIC %md
# MAGIC ## Cell 4: Ingest WorldPop Population

local_pop_path = os.path.join(os.getcwd(), "worldpop_bkk.tif")

lat_steps = np.arange(BBOX["south"], BBOX["north"], GRID_LAT_SIZE)
lon_steps = np.arange(BBOX["west"],  BBOX["east"],  GRID_LON_SIZE)
coords    = [(lon, lat) for lat in lat_steps for lon in lon_steps]

pop_records = []
print(f"⏳ Sampling population density from: {local_pop_path}")

try:
    with rasterio.open(local_pop_path) as src:
        for coord, val in zip(coords, src.sample(coords)):
            pop_density = max(float(val[0]), 0.0)  # clamp NoData → 0
            pop_records.append({
                "grid_lat":    round(coord[1], 4),
                "grid_lon":    round(coord[0], 4),
                "pop_density": pop_density,
            })

    df_pop = spark.createDataFrame(pd.DataFrame(pop_records)) \
                  .withColumn("ingested_at", current_timestamp())
    df_pop.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{DATABASE_NAME}.bronze_population")
    print(f"✅ Saved {df_pop.count():,} records → bronze_population")
    display(df_pop.limit(5))

except Exception as e:
    print(f"❌ Error: {e}")

⏳ Sampling population density from: /Workspace/Users/kittikhunforwork@gmail.com/EV_Chage/worldpop_bkk.tif
✅ Saved 18,492 records → bronze_population


grid_lat,grid_lon,pop_density,ingested_at
13.45,100.25,0.0,2026-05-11T23:57:23.273Z
13.45,100.2551,0.0,2026-05-11T23:57:23.273Z
13.45,100.2602,0.0,2026-05-11T23:57:23.273Z
13.45,100.2653,0.0,2026-05-11T23:57:23.273Z
13.45,100.2704,0.0,2026-05-11T23:57:23.273Z


Cell  5: Ingest Flood Risk Data

In [0]:
# MAGIC %md
# MAGIC ## Cell 5: Ingest Flood Risk Data

local_flood_path = os.path.join(os.getcwd(), "flood_bangkok.tif")

flood_records = []
print(f"⏳ Sampling flood risk from: {local_flood_path}")

try:
    with rasterio.open(local_flood_path) as src:
        for coord, val in zip(coords, src.sample(coords)):
            flood_occ = max(float(val[0]), 0.0)  # clamp NoData → 0
            flood_records.append({
                "grid_lat":         round(coord[1], 4),
                "grid_lon":         round(coord[0], 4),
                "flood_occurrence": flood_occ,
            })

    df_flood = spark.createDataFrame(pd.DataFrame(flood_records)) \
                    .withColumn("ingested_at", current_timestamp())
    df_flood.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{DATABASE_NAME}.bronze_flood_risk")
    print(f"✅ Saved {df_flood.count():,} records → bronze_flood_risk")
    display(df_flood.limit(5))

except Exception as e:
    print(f"❌ Error: {e}")

⏳ Sampling flood risk from: /Workspace/Users/kittikhunforwork@gmail.com/EV_Chage/flood_bangkok.tif
✅ Saved 18,492 records → bronze_flood_risk


grid_lat,grid_lon,flood_occurrence,ingested_at
13.45,100.25,100.0,2026-05-11T23:57:29.254Z
13.45,100.2551,99.0,2026-05-11T23:57:29.254Z
13.45,100.2602,100.0,2026-05-11T23:57:29.254Z
13.45,100.2653,99.0,2026-05-11T23:57:29.254Z
13.45,100.2704,100.0,2026-05-11T23:57:29.254Z


Cell 6: Master Grid

In [0]:
# MAGIC %md
# MAGIC ## Cell 6: Create Master Grid

lat_steps   = np.arange(BBOX["south"], BBOX["north"], GRID_LAT_SIZE)
lon_steps   = np.arange(BBOX["west"],  BBOX["east"],  GRID_LON_SIZE)
master_grid = [
    {"grid_lat": round(float(la), 4), "grid_lon": round(float(lo), 4)}
    for la in lat_steps for lo in lon_steps
]

df_master = spark.createDataFrame(pd.DataFrame(master_grid))
df_master.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{DATABASE_NAME}.master_grid")

print(f"✅ Master Grid created: {df_master.count():,} cells")

✅ Master Grid created: 18,492 cells


Cell 7: Data Integrity Check

In [0]:
# MAGIC %md
# MAGIC ## Cell 7: Data Integrity Check

from pyspark.sql.functions import col

tables_to_check = ["master_grid", "bronze_osm_poi", "bronze_population", "bronze_flood_risk"]

print(f"📊 Ingestion Summary — Database: {DATABASE_NAME}\n")
print(f"{'Table':<22} | {'Status':<12} | Records")
print("-" * 52)

for table in tables_to_check:
    try:
        count  = spark.table(f"{DATABASE_NAME}.{table}").count()
        status = "✅ Ready" if count > 0 else "⚠️ Empty"
        print(f"{table:<22} | {status:<12} | {count:,}")
    except Exception:
        print(f"{table:<22} | ❌ Missing   | 0")

print("-" * 52)

try:
    display(spark.table(f"{DATABASE_NAME}.bronze_osm_poi")
                 .groupBy("poi_type").count()
                 .orderBy(col("count").desc()))
except Exception:
    print("No POI data found.")

try:
    display(spark.table(f"{DATABASE_NAME}.master_grid").selectExpr(
        "min(grid_lat) as min_lat", "max(grid_lat) as max_lat",
        "min(grid_lon) as min_lon", "max(grid_lon) as max_lon"
    ))
except Exception:
    print("No Master Grid data found.")

📊 Ingestion Summary — Database: ev_site_selection

Table                  | Status       | Records
----------------------------------------------------
master_grid            | ✅ Ready      | 18,492
bronze_osm_poi         | ✅ Ready      | 2,453
bronze_population      | ✅ Ready      | 18,492
bronze_flood_risk      | ✅ Ready      | 18,492
----------------------------------------------------


poi_type,count
office_building,893
hotel,692
fuel_station,472
other,310
charging_station,59
shopping_mall,27


min_lat,max_lat,min_lon,max_lon
13.45,14.0485,100.25,100.9487
